### 1_2 - Preprocessing to save one dataset for  geometric + semantic info based on query Column

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [23]:
data = pd.read_parquet('ds/shopping_queries_dataset_examples.parquet')

In [24]:
#to focus only on one locale like us

data = data[data['product_locale'] == 'us']
data = data.sample(n=50000, random_state=42).reset_index(drop=True)

#data info

print(f"shape: {data.shape}")
print()
print(data.dtypes)

shape: (50000, 9)

example_id         int64
query             object
query_id           int64
product_id        object
product_locale    object
esci_label        object
small_version      int64
large_version      int64
split             object
dtype: object


In [25]:
data.head(2)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,2219601,workout weights,113894,B07HGX6G9K,us,E,1,1,train
1,1452229,nintendo gift cards,73849,B00K59HKIQ,us,I,0,1,train


In [26]:
#drop unessery columns since we don t use other dataset

# and also since we d ont want to use product dataset and our goal is only categorise based on query it self we
# don t need to use esci (exact , , , ....) labelling
data = data.drop(columns=['example_id' , 'product_id' , 'query_id' , 'split' , 'product_locale' ,'esci_label' , 'small_version' , 'large_version'])

In [27]:
data.info()

#we have now only one columns query

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   query   50000 non-null  object
dtypes: object(1)
memory usage: 390.8+ KB


In [ ]:
#convert obj to string 
data['query'] = data['query'].astype('string')

In [29]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   query   50000 non-null  string
dtypes: string(1)
memory usage: 390.8 KB


In [30]:
data.isnull().sum()

query    0
dtype: int64

In [ ]:
#geometrique Columns

# new columns from str columns
data['query_chars'] = data['query'].fillna('').str.len()
data['query_words'] = data['query'].fillna('').str.split().str.len()
#contain a number digit it could define or segment people the one that search for specifique like tv 50 inch 0 or  1
data['contains_digit'] = data['query'].str.contains(r'\d').astype(int)
#count digit
data['digit_count'] = data['query'].str.count(r'\d')

#average word length
data['avg_word_len'] = data['query_chars'] / data['query_words'].replace(0, 1)

#space count
data['space_count'] = data['query'].str.count(' ')

#special cracters count
data['spec_char_count'] = data['query'].str.count(r'[^a-zA-Z0-9\s]')

#upper case ratio no need everything low case
#data['upper_ratio'] = data['query'].str.findall(r'[A-Z]').str.len() / data['query_chars']


#since we are based on just query we tried to extract max info 

In [ ]:
#semantic Column 

from sentence_transformers import SentenceTransformer

#it s a model that take a sentence and convert it to a vector of many dimmensions like dim1 take sentiment dim2 verb tense ....

#and this model contains 384 dimmension
# so simply it s a model takes text and convert to a numbre vector a meaning in math
#it s pretrained model with neural network


model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

embeddings = model.encode(
    data['query'].tolist(),
    batch_size=256,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

In [33]:
embeddings.shape 

(50000, 384)

In [ ]:
from sklearn.decomposition import PCA

# reduce deminsions of embidings from 384 to 50

pca = PCA(n_components=50, random_state=42)
emb_reduced = pca.fit_transform(embeddings)

In [35]:
emb_reduced

array([[-0.02886857,  0.04139109,  0.14737184, ..., -0.0187175 ,
        -0.05028488, -0.04530716],
       [ 0.04060929, -0.08341733, -0.03375222, ...,  0.00740021,
         0.08707064, -0.10552342],
       [-0.05210952,  0.21255182, -0.00198069, ...,  0.02381836,
         0.01264411,  0.04207192],
       ...,
       [ 0.15693039,  0.08208084, -0.05545574, ...,  0.0497801 ,
         0.06580137,  0.06661241],
       [ 0.04475829,  0.10749577, -0.0687483 , ...,  0.02444487,
         0.01034996,  0.09347254],
       [ 0.06997514,  0.15411761,  0.00182331, ..., -0.09940969,
        -0.0298657 ,  0.00399213]], shape=(50000, 50), dtype=float32)

In [ ]:
#save it to dataframe

emb_df = pd.DataFrame(
    emb_reduced, 
    columns=[f'emb_{i}' for i in range(emb_reduced.shape[1])]
)

In [ ]:
geo_features = [
    'query_chars', 'query_words', 'contains_digit',
    'digit_count', 'avg_word_len', 'space_count', 'spec_char_count'
]

In [44]:
from sklearn.preprocessing import StandardScaler

geo_cols = ['query_chars', 'query_words', 'contains_digit', 'digit_count' , 'avg_word_len' , 'space_count' , 'spec_char_count'] 

scaler = StandardScaler()
data[geo_cols] = scaler.fit_transform(data[geo_cols])

In [ ]:
#save all to one dataframe

final_data = pd.concat(
    [data , emb_df],
    axis=1
)

In [46]:
final_data.head(8)

,query,query_chars,query_words,contains_digit,digit_count,avg_word_len,space_count,spec_char_count,emb_0,emb_1,...,emb_40,emb_41,emb_42,emb_43,emb_44,emb_45,emb_46,emb_47,emb_48,emb_49
0,workout weights,-0.651438,-0.917934,-0.419985,-0.365241,1.004975,-0.913646,-0.201665,-0.028869,0.041391,...,-0.091584,0.011627,0.097332,0.024175,-0.122666,0.036653,-0.021834,-0.018718,-0.050285,-0.045307
1,nintendo gift cards,-0.250755,-0.346518,-0.419985,-0.365241,0.160833,-0.345260,-0.201665,0.040609,-0.083417,...,-0.046814,0.022521,0.077184,0.015270,-0.049487,0.060322,-0.030696,0.007400,0.087071,-0.105523
2,urban skin rx,-0.851779,-0.346518,-0.419985,-0.365241,-1.286269,-0.345260,-0.201665,-0.052110,0.212552,...,-0.040206,-0.162135,0.078060,0.077822,-0.008254,-0.028601,0.086238,0.023818,0.012644,0.042072
3,bird seed,-1.252462,-0.917934,-0.419985,-0.365241,-1.165677,-0.913646,-0.201665,0.067840,0.210649,...,-0.028385,0.142146,-0.130093,-0.177015,-0.045551,0.017446,-0.038451,-0.157676,0.060101,0.037510
4,+foot cream without alcohol,0.550610,0.224898,-0.419985,-0.365241,0.462312,0.223126,2.753563,-0.035556,0.025388,...,0.053627,-0.013516,-0.142906,-0.005557,0.036538,0.171148,-0.047727,0.135562,-0.066205,-0.138394
5,brother tn730 high yield black toner,1.452146,1.367730,2.381036,3.337777,-0.080351,1.359897,-0.201665,0.244192,0.000418,...,-0.041128,0.011045,0.007904,0.013253,0.139528,0.044885,-0.072630,0.071854,0.073138,0.062711
6,60 lashes,-1.252462,-0.917934,2.381036,2.103437,-1.165677,-0.913646,-0.201665,-0.069236,0.075436,...,-0.088331,0.072093,0.010556,-0.046400,-0.002193,-0.065785,-0.075137,0.130785,0.090734,-0.059412
7,cpap filters,-0.951950,-0.917934,-0.419985,-0.365241,-0.080351,-0.913646,-0.201665,0.095043,-0.103721,...,-0.092402,0.048831,0.191065,-0.041545,-0.117167,0.086743,0.033624,0.089814,-0.010352,0.148334


In [47]:
final_data.to_csv("ds/cleaned_amazon_queries_dataset_en_50k_geo_semant.csv")